# My tables: studios, directors, cast, reviewers, user_reviews

**Author: Paulius Sivickas**

Database Management, AUAS — individual contribution.

Builds into PostgreSQL database `Database_Assignemtnt_DDB`.

Run the cells top to bottom. Cell 2 asks for your Postgres password once; it is not saved in the file.

## 1. Packages

In [25]:
%pip install psycopg2-binary openpyxl --quiet
print("done")

Note: you may need to restart the kernel to use updated packages.
done


## 2. Connect

`getpass` hides the password and keeps it out of the saved notebook, so nothing secret ends up on GitHub.

In [26]:
# Author: Paulius Sivickas
import os, re, getpass
import pandas as pd
import psycopg2

PASSWORD = getpass.getpass("Postgres password: ")

con = psycopg2.connect(
    dbname="Database_Assignemtnt_DDB",
    user="postgres",
    password=PASSWORD,
    host="localhost",
    port="5432",
)
cur = con.cursor()
cur.execute("SELECT version();")
print(cur.fetchone()[0])

PostgreSQL 18.6 on aarch64-apple-darwin24.6.0, compiled by Apple clang version 17.0.0 (clang-1700.0.13.5), 64-bit


## 3. Check the data files

In [27]:
DATA = "data"
files = {
    "meta":   os.path.join(DATA, "metaClean43Brightspace.xlsx"),
    "sales":  os.path.join(DATA, "sales.xlsx"),
    "user":   os.path.join(DATA, "UserReviewsClean43LIWC.xlsx"),
    "expert": os.path.join(DATA, "ExpertReviewsClean43LIWC.xlsx"),
}
for k, p in files.items():
    print(("  OK      " if os.path.exists(p) else "  MISSING "), k, "->", p)

HAVE_USER_REVIEWS = os.path.exists(files["user"])
if not HAVE_USER_REVIEWS:
    print("\nUserReviews file not present - the user_reviews TABLE will still be created, just left empty.")

  OK       meta -> data/metaClean43Brightspace.xlsx
  OK       sales -> data/sales.xlsx
  OK       user -> data/UserReviewsClean43LIWC.xlsx
  MISSING  expert -> data/ExpertReviewsClean43LIWC.xlsx


## 4. Create the tables

`movies` is here because my tables point at it with foreign keys, so it has to exist first.

**My tables:** `studios`, `directors`, `cast_members`, `movie_cast`, `reviewers`, `user_reviews`.

Two design points I can defend:

- `studios` and `directors` are separate tables because the studio name repeated on every movie row is a **3NF** problem — the name depends on the studio, not on the movie.
- `cast` holds up to 20 comma-separated names in one cell. That breaks **1NF**, so it becomes `cast_members` + `movie_cast`, a many-to-many (the deck's Rule 5, case 1). `billing_order` is the relationship's own attribute, which is Rule 4.

In [28]:
# Author: Paulius Sivickas
SCHEMA = """
DROP TABLE IF EXISTS movie_cast    CASCADE;
DROP TABLE IF EXISTS user_reviews  CASCADE;
DROP TABLE IF EXISTS cast_members  CASCADE;
DROP TABLE IF EXISTS reviewers     CASCADE;
DROP TABLE IF EXISTS movies        CASCADE;
DROP TABLE IF EXISTS studios       CASCADE;
DROP TABLE IF EXISTS directors     CASCADE;

CREATE TABLE studios (
    studio_id   SERIAL PRIMARY KEY,
    studio_name TEXT NOT NULL UNIQUE
);

CREATE TABLE directors (
    director_id   SERIAL PRIMARY KEY,
    director_name TEXT NOT NULL UNIQUE
);

CREATE TABLE movies (
    movie_id             INTEGER PRIMARY KEY,
    metacritic_url       TEXT NOT NULL UNIQUE,
    title                TEXT,
    release_date         DATE,
    runtime_minutes      NUMERIC,
    mpaa_rating          TEXT,
    production_budget    NUMERIC,
    box_office_worldwide NUMERIC,
    box_office_domestic  NUMERIC,
    studio_id            INTEGER REFERENCES studios(studio_id),
    director_id          INTEGER REFERENCES directors(director_id)
);

CREATE TABLE cast_members (
    person_id   SERIAL PRIMARY KEY,
    person_name TEXT NOT NULL UNIQUE
);

CREATE TABLE movie_cast (
    movie_id      INTEGER NOT NULL REFERENCES movies(movie_id),
    person_id     INTEGER NOT NULL REFERENCES cast_members(person_id),
    billing_order INTEGER,
    PRIMARY KEY (movie_id, person_id)
);

CREATE TABLE reviewers (
    reviewer_id   SERIAL PRIMARY KEY,
    reviewer_name TEXT NOT NULL,
    reviewer_type TEXT NOT NULL CHECK (reviewer_type IN ('user','expert')),
    UNIQUE (reviewer_name, reviewer_type)
);

CREATE TABLE user_reviews (
    review_id              INTEGER PRIMARY KEY,
    movie_id               INTEGER NOT NULL REFERENCES movies(movie_id),
    reviewer_id            INTEGER REFERENCES reviewers(reviewer_id),
    review_date            DATE,
    review_score           NUMERIC,
    review_text            TEXT,
    thumbs_up              INTEGER,
    total_thumbs           INTEGER,
    word_count             NUMERIC,
    words_per_sentence     NUMERIC,
    pronoun_count          NUMERIC,
    possessive_pronoun_pct NUMERIC,
    positive_emotion_pct   NUMERIC,
    negative_emotion_pct   NUMERIC,
    analytic               NUMERIC,
    clout                  NUMERIC,
    authentic              NUMERIC,
    tone                   NUMERIC,
    certain_pct            NUMERIC,
    tentative_pct          NUMERIC
);
"""
cur.execute(SCHEMA)
con.commit()

cur.execute("""SELECT table_name FROM information_schema.tables
               WHERE table_schema='public' ORDER BY table_name;""")
for (t,) in cur.fetchall():
    print("created:", t)

created: cast_members
created: directors
created: movie_cast
created: movies
created: reviewers
created: studios
created: user_reviews


## 5. Little helpers

In [29]:
# Author: Paulius Sivickas
def clean_text(v):
    """Strip the stray quotes and spaces the scraper left behind."""
    if v is None: return None
    s = str(v).strip()
    if s in ("None", "nan", ""): return None
    return s.strip("'\"").strip() or None

def split_list(v):
    """'Drama,Crime' -> ['Drama','Crime']"""
    s = clean_text(v)
    return [p.strip() for p in s.split(",") if p.strip()] if s else []

def norm_title(t):
    """Lowercase, drop a trailing '(S-Korea)', strip punctuation. Used to match sales to metaClean."""
    if t is None or pd.isna(t): return None
    s = re.sub(r"\s*\(.*?\)\s*$", "", str(t).lower().strip())
    return re.sub(r"[^a-z0-9]+", "", s) or None

def num(v):
    try:
        if v is None or (isinstance(v, float) and pd.isna(v)): return None
        return float(v)
    except (TypeError, ValueError):
        return None

print("helpers ready")

helpers ready


## 6. Read the spreadsheets

In [30]:
meta  = pd.read_excel(files["meta"])
sales = pd.read_excel(files["sales"], sheet_name="numbers253")
sales = sales.drop(columns=[c for c in sales.columns if str(c).startswith("Unnamed")])

print("metaClean:", meta.shape)
print("sales    :", sales.shape)

sales["title_norm"] = sales["title"].map(norm_title)
sales_best = (sales.sort_values("worldwide_box_office", ascending=False)
                   .drop_duplicates("title_norm", keep="first")
                   .set_index("title_norm"))
print("unique sales titles:", len(sales_best))

metaClean: (11364, 13)
sales    : (30612, 15)
unique sales titles: 29905


## 7. studios and directors — **my tables**

One row per studio, one row per director. The film points at them.

About 1.7% of films list two studios (co-productions) and 0.03% list two directors. Only the first is used. That is a documented limitation.

In [31]:
# Author: Paulius Sivickas
studio_names   = sorted({split_list(v)[0] for v in meta["studio"].dropna()   if split_list(v)})
director_names = sorted({split_list(v)[0] for v in meta["director"].dropna() if split_list(v)})

cur.executemany("INSERT INTO studios (studio_name) VALUES (%s)",   [(n,) for n in studio_names])
cur.executemany("INSERT INTO directors (director_name) VALUES (%s)", [(n,) for n in director_names])
con.commit()

cur.execute("SELECT studio_name, studio_id FROM studios")
studio_ids = dict(cur.fetchall())
cur.execute("SELECT director_name, director_id FROM directors")
director_ids = dict(cur.fetchall())

print("studios  :", len(studio_ids))
print("directors:", len(director_ids))

studios  : 1117
directors: 6136


## 8. movies (needed by my foreign keys)

In [32]:
movie_rows, url_to_id = [], {}
for i, r in enumerate(meta.itertuples(index=False), start=1):
    url = clean_text(r.url)
    if not url or url in url_to_id:
        continue
    url_to_id[url] = i

    tn = norm_title(r.title)
    s  = sales_best.loc[tn] if tn in sales_best.index else None

    rating = clean_text(r.rating)
    if rating:
        rating = rating.lstrip("| ").strip()     # "| PG-13" -> "PG-13"

    st = split_list(r.studio)
    dr = split_list(r.director)
    rel = r.RelDate if not pd.isna(r.RelDate) else None

    movie_rows.append((
        i, url, clean_text(r.title), rel, num(r.runtime), rating,
        num(s["production_budget"])    if s is not None else None,
        num(s["worldwide_box_office"]) if s is not None else None,
        num(s["domestic_box_office"])  if s is not None else None,
        studio_ids.get(st[0])   if st else None,
        director_ids.get(dr[0]) if dr else None,
    ))

cur.executemany("""INSERT INTO movies
    (movie_id, metacritic_url, title, release_date, runtime_minutes, mpaa_rating,
     production_budget, box_office_worldwide, box_office_domestic, studio_id, director_id)
    VALUES (%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s)""", movie_rows)
con.commit()

print("movies:", len(movie_rows))
print("  with budget        :", sum(1 for m in movie_rows if m[6] is not None))
print("  with worldwide BO  :", sum(1 for m in movie_rows if m[7] is not None))
print("  with domestic BO   :", sum(1 for m in movie_rows if m[8] is not None))

movies: 11364
  with budget        : 3847
  with worldwide BO  : 5952
  with domestic BO   : 7540


## 9. cast_members and movie_cast — **my tables**

`cast` has commas in 94.2% of rows, up to 20 names in one cell. That is the 1NF fix.

In [33]:
# Author: Paulius Sivickas
people = sorted({n for v in meta["cast"].dropna() for n in split_list(v)})
cur.executemany("INSERT INTO cast_members (person_name) VALUES (%s)", [(n,) for n in people])
con.commit()

cur.execute("SELECT person_name, person_id FROM cast_members")
person_ids = dict(cur.fetchall())

mc_rows, seen = [], set()
for r in meta.itertuples(index=False):
    mid = url_to_id.get(clean_text(r.url))
    if mid is None: continue
    for order, name in enumerate(split_list(r.cast), start=1):
        pid = person_ids.get(name)
        if pid is None or (mid, pid) in seen:
            continue                       # same actor listed twice for one film
        seen.add((mid, pid))
        mc_rows.append((mid, pid, order))

cur.executemany("INSERT INTO movie_cast (movie_id, person_id, billing_order) VALUES (%s,%s,%s)",
                mc_rows)
con.commit()

print("cast_members:", len(person_ids))
print("movie_cast  :", len(mc_rows))
print("films with no cast listed:", len(url_to_id) - len({m[0] for m in mc_rows}))

cast_members: 24861
movie_cast  : 48284
films with no cast listed: 3702


## 10. reviewers and user_reviews — **my tables**

The file is 242 MB, so this streams it row by row instead of loading it all into memory.

`positive_emotion_pct` / `negative_emotion_pct` come from LIWC's `posemo` / `negemo`. The draw.io sketch calls them *pronoun* columns, which is wrong — LIWC has no positive pronoun.

`LIMIT` keeps the first run short. Set it to `None` for everything.

In [34]:
# Author: Paulius Sivickas
LIMIT = None        # set to None for all ~320,000 rows

if not HAVE_USER_REVIEWS:
    print("UserReviews file missing - table created but left empty.")
else:
    import openpyxl
    wb = openpyxl.load_workbook(files["user"], read_only=True)
    ws = wb[wb.sheetnames[0]]
    rows = ws.iter_rows(values_only=True)
    hdr  = {name: i for i, name in enumerate(next(rows))}
    g = lambda row, c: row[hdr[c]] if c in hdr and hdr[c] < len(row) else None

    reviewer_ids, batch, rid, kept, skipped = {}, [], 0, 0, 0
    LIWC = ["WC","WPS","pronoun","ppron","posemo","negemo",
            "Analytic","Clout","Authentic","Tone","certain","tentat"]

    for row in rows:
        if LIMIT is not None and kept >= LIMIT:
            break
        mid = url_to_id.get(clean_text(g(row, "url")))
        if mid is None:
            skipped += 1                      # review for a film not in metaClean
            continue
        rid += 1; kept += 1

        rname = clean_text(g(row, "reviewer"))
        if rname and rname not in reviewer_ids:
            reviewer_ids[rname] = None

        d = g(row, "dateP")
        d = pd.to_datetime(clean_text(d), errors="coerce")
        d = None if pd.isna(d) else d.date()

        batch.append([rid, mid, rname, d,
                      num(g(row, "idvscore")), clean_text(g(row, "Rev")),
                      num(g(row, "thumbsUp")), num(g(row, "thumbsTot"))]
                     + [num(g(row, c)) for c in LIWC])
    wb.close()

    # reviewers first, so user_reviews.reviewer_id has something to point at
    names = sorted(reviewer_ids)
    cur.executemany("INSERT INTO reviewers (reviewer_name, reviewer_type) VALUES (%s,'user')",
                    [(n,) for n in names])
    con.commit()
    cur.execute("SELECT reviewer_name, reviewer_id FROM reviewers WHERE reviewer_type='user'")
    reviewer_ids = dict(cur.fetchall())

    for b in batch:
        b[2] = reviewer_ids.get(b[2])         # name -> id

    cols = ("review_id, movie_id, reviewer_id, review_date, review_score, review_text, "
            "thumbs_up, total_thumbs, word_count, words_per_sentence, pronoun_count, "
            "possessive_pronoun_pct, positive_emotion_pct, negative_emotion_pct, "
            "analytic, clout, authentic, tone, certain_pct, tentative_pct")
    cur.executemany(f"INSERT INTO user_reviews ({cols}) VALUES ({','.join(['%s']*20)})", batch)
    con.commit()

    print(f"reviewers   : {len(reviewer_ids):,}")
    print(f"user_reviews: {kept:,} loaded, {skipped:,} skipped (film not in metaClean)")

reviewers   : 106,792
user_reviews: 319,662 loaded, 0 skipped (film not in metaClean)


## 11. Check it worked

In [35]:
print("ROW COUNTS")
for t in ["studios","directors","movies","cast_members","movie_cast","reviewers","user_reviews"]:
    cur.execute(f"SELECT COUNT(*) FROM {t}")
    print(f"  {t:<14} {cur.fetchone()[0]:>10,}")

print("\nTOP 10 STUDIOS BY NUMBER OF FILMS")
q = """
SELECT s.studio_name, COUNT(*) AS films,
       ROUND(AVG(m.box_office_domestic)) AS avg_domestic_bo
FROM movies m
JOIN studios s ON s.studio_id = m.studio_id
GROUP BY s.studio_name
ORDER BY films DESC
LIMIT 10
"""
print(pd.read_sql_query(q, con).to_string(index=False))

print("\nTOP 10 MOST-CAST ACTORS")
q2 = """
SELECT p.person_name, COUNT(*) AS films
FROM movie_cast mc
JOIN cast_members p ON p.person_id = mc.person_id
GROUP BY p.person_name
ORDER BY films DESC
LIMIT 10
"""
print(pd.read_sql_query(q2, con).to_string(index=False))

ROW COUNTS
  studios             1,117
  directors           6,136
  movies             11,364
  cast_members       24,861
  movie_cast         48,284
  reviewers         106,792
  user_reviews      319,662

TOP 10 STUDIOS BY NUMBER OF FILMS
                           studio_name  films  avg_domestic_bo
                             IFC Films    408        1718878.0
                    Universal Pictures    345       73725841.0
                Sony Pictures Classics    344        3603493.0
                 Warner Bros. Pictures    302       69178080.0
                               Netflix    301        5721147.0
                    Paramount Pictures    280       75863577.0
                     Magnolia Pictures    268        1037711.0
Twentieth Century Fox Film Corporation    268       72321821.0
                     Columbia Pictures    252       81573913.0
                             Lionsgate    239       34095985.0

TOP 10 MOST-CAST ACTORS
      person_name  films
Samuel L. Jacks

/var/folders/0y/1vkvn6y96hd9_k1830qdmbvh0000gn/T/ipykernel_21564/3794495595.py:16: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  print(pd.read_sql_query(q, con).to_string(index=False))
/var/folders/0y/1vkvn6y96hd9_k1830qdmbvh0000gn/T/ipykernel_21564/3794495595.py:27: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  print(pd.read_sql_query(q2, con).to_string(index=False))


In [36]:
con.commit()
con.close()
print("finished")

finished
